# BuzzSpot Challenge — Pipeline End-to-End

**CVPPA @ ECCV 2026 — Détection de pollinisateurs en vidéo**

1. Environnement & GPU
2. Installation des dépendances
3. Clonage du repo
4. Téléchargement & extraction du dataset BuzzSet
5. Exploration & statistiques
6. (Optionnel) Masques SAM 2 pour copy-paste
7. Entraînement (RF-DETR, config T4 ou L4)
8. Évaluation COCO (mAP@0.5:0.95)
9. Inférence tuilée + TTA + WBF sur le jeu de test
10. Génération de la soumission `.zip`
11. (Optionnel) Pseudo-labeling + self-training

---
**Repo** : https://github.com/hashirama21/buzzspot-gik  
**Dataset** : https://phenoroam.phenorob.de/file-uploader/download/public/35261367058-BuzzSet_challenge.zip

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hashirama21/buzzspot-gik/blob/main/notebooks/buzzspot_pipeline.ipynb)

## 0 — Environnement & GPU

In [ ]:
import sys, os, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Détection GPU
try:
    gpu_name = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True
    ).strip().split('\n')[0]
except Exception:
    gpu_name = ''

if not gpu_name:
    print('ATTENTION : aucun GPU détecté — entraînement sur CPU, extrêmement lent.')
    print('Dans Colab : Exécution → Modifier le type d\'exécution → GPU T4, puis reconnecter.')
    GPU_TIER = 'cpu'
elif any(x in gpu_name for x in ('A100', 'H100')):
    print(f'GPU détecté : {gpu_name}  → tier A100/H100')
    GPU_TIER = 'a100'
elif any(x in gpu_name for x in ('L4', 'L40', 'A10')):
    print(f'GPU détecté : {gpu_name}  → tier L4')
    GPU_TIER = 'l4'
elif 'T4' in gpu_name:
    print(f'GPU détecté : {gpu_name}  → tier T4')
    GPU_TIER = 't4'
else:
    print(f'GPU détecté : {gpu_name}  → tier inconnu, utilisation config T4')
    GPU_TIER = 't4'

# Config name Hydra correspondante
CONFIG_NAME = {
    'a100': 'colab_l4',   # L4 config convient aussi pour A100 (ajuster batch si besoin)
    'l4':   'colab_l4',
    't4':   'colab_t4',
    'cpu':  'default',
}[GPU_TIER]
print(f'Config Hydra → {CONFIG_NAME}')

## 1 — Installation des dépendances

In [ ]:
# cu121 pour CUDA 12.1 (T4/L4 Colab) ; remplacer par cu118 ou cpu si besoin
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

!pip install -q --upgrade \
    rfdetr \
    'albumentations>=2.0.8,<3.0.0' \
    ensemble-boxes \
    opencv-python-headless \
    pycocotools \
    hydra-core \
    omegaconf \
    scipy \
    numpy \
    pandas \
    wandb \
    tqdm \
    rich \
    matplotlib \
    seaborn

print('Installation terminée')

## 2 — WandB (optionnel)

In [ ]:
# Coller votre clé API depuis https://wandb.ai/authorize
# Laisser vide pour désactiver WandB (les métriques s'affichent quand même dans le terminal).
WANDB_API_KEY = ''  # ex : 'abcdef1234567890abcdef1234567890abcdef12'

import os
if WANDB_API_KEY:
    os.environ['WANDB_API_KEY'] = WANDB_API_KEY
    import wandb
    wandb.login(key=WANDB_API_KEY, relogin=True)
    print('WandB connecté.')
else:
    os.environ['WANDB_MODE'] = 'disabled'
    print('WandB désactivé — métriques dans le terminal uniquement.')

## 3 — Clonage du repo

In [ ]:
REPO_URL = 'https://github.com/hashirama21/buzzspot-gik.git'
REPO_DIR = '/content/buzzspot'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Répertoire courant :', os.getcwd())

## 4 — Téléchargement et extraction du dataset BuzzSet

In [ ]:
import zipfile, urllib.request
from pathlib import Path
from tqdm.notebook import tqdm

DATASET_URL  = 'https://phenoroam.phenorob.de/file-uploader/download/public/35261367058-BuzzSet_challenge.zip'
ZIP_PATH     = Path('data/BuzzSet_challenge.zip')
EXTRACT_ROOT = Path('data/')
BUZZSET_DIR  = EXTRACT_ROOT / 'buzzset'

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    print(f'Téléchargement …')

    class _Progress(tqdm):
        def update_to(self, b=1, bsize=1, tsize=None):
            if tsize is not None:
                self.total = tsize
            self.update(b * bsize - self.n)

    with _Progress(unit='B', unit_scale=True, unit_divisor=1024, miniters=1, desc='BuzzSet') as pb:
        urllib.request.urlretrieve(DATASET_URL, ZIP_PATH, reporthook=pb.update_to)
    print(f'Archive sauvegardée : {ZIP_PATH}  ({ZIP_PATH.stat().st_size / 1e6:.0f} MB)')
else:
    print(f'Archive déjà présente : {ZIP_PATH}')

if not BUZZSET_DIR.exists():
    print('Extraction …')
    with zipfile.ZipFile(ZIP_PATH) as z:
        for m in tqdm(z.namelist(), desc='Extraction'):
            z.extract(m, EXTRACT_ROOT)
    extracted_dirs = [
        p for p in EXTRACT_ROOT.iterdir()
        if p.is_dir() and p.name not in ('buzzset', '__MACOSX') and (p / 'annotations').exists()
    ]
    if extracted_dirs and not BUZZSET_DIR.exists():
        extracted_dirs[0].rename(BUZZSET_DIR)
    print('Extraction terminée')
else:
    print('Dataset déjà extrait')

In [ ]:
SPLIT_IMG_DIR = {'train': 'train', 'valid': 'valid', 'test': 'test_devphase'}
for split in ('train', 'valid', 'test'):
    ann = EXTRACT_ROOT / 'buzzset' / 'annotations' / f'{split}.json'
    img = EXTRACT_ROOT / 'buzzset' / SPLIT_IMG_DIR[split]
    status = 'OK' if ann.exists() and img.exists() else 'MANQUANT'
    print(f'[{status}] {split:5s} — annotations: {ann.exists()}, images: {img.exists()}')

## 5 — Exploration du dataset

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

CLASS_NAMES = {1: 'bee', 2: 'bumblebee', 3: 'hoverfly', 4: 'moth'}
SPLITS = {}
for split in ('train', 'valid', 'test'):
    p = EXTRACT_ROOT / 'buzzset' / 'annotations' / f'{split}.json'
    if p.exists():
        with open(p) as f:
            SPLITS[split] = json.load(f)

for split, data in SPLITS.items():
    print(f'{split:6s} : {len(data["images"]):5d} images, {len(data.get("annotations", [])):6d} annotations')

In [ ]:
if 'train' in SPLITS:
    anns   = SPLITS['train']['annotations']
    counts = {name: 0 for name in CLASS_NAMES.values()}
    blur_count = occ_count = 0

    for ann in anns:
        counts[CLASS_NAMES.get(ann['category_id'], 'unknown')] = \
            counts.get(CLASS_NAMES.get(ann['category_id'], 'unknown'), 0) + 1
        attrs = ann.get('attributes', {})
        blur_count += attrs.get('blur', 0)
        occ_count  += attrs.get('occlusion', 0)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    ax = axes[0]
    bars = ax.bar(list(counts.keys()), list(counts.values()),
                  color=sns.color_palette('muted', len(counts)))
    ax.bar_label(bars)
    ax.set_title('Distribution des classes (train)')
    ax.set_ylabel('Instances')

    ax = axes[1]
    total = len(anns)
    ax.bar(['Blur', 'Occlusion', 'Propre'],
           [blur_count, occ_count, total - blur_count - occ_count],
           color=['#e74c3c', '#e67e22', '#2ecc71'])
    ax.set_title('Attributs de dégradation')
    ax.set_ylabel('Instances')

    plt.tight_layout()
    plt.show()
    print(f'Total : {total} | Blur : {blur_count} ({100*blur_count/total:.1f}%) | Occlusion : {occ_count} ({100*occ_count/total:.1f}%)')

In [ ]:
if 'train' in SPLITS:
    areas = [ann['area'] for ann in SPLITS['train']['annotations']]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(np.sqrt(areas), bins=60, color='steelblue', edgecolor='white', linewidth=0.5)
    ax.axvline(32, color='r',      linestyle='--', label='Small (32px)')
    ax.axvline(96, color='orange', linestyle='--', label='Medium (96px)')
    ax.set_xlabel('√aire (pixels)')
    ax.set_ylabel('Fréquence')
    ax.set_title('Distribution des tailles d\'instances')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
import cv2, random

COLORS = {1: (52,152,219), 2: (231,76,60), 3: (46,204,113), 4: (155,89,182)}

def show_annotated_sample(split='train', n=4, seed=42):
    data    = SPLITS[split]
    id2img  = {img['id']: img for img in data['images']}
    id2anns = {}
    for ann in data.get('annotations', []):
        id2anns.setdefault(ann['image_id'], []).append(ann)

    rng    = random.Random(seed)
    sample = rng.sample([iid for iid in id2anns if id2anns[iid]], min(n, len(id2anns)))

    fig, axes = plt.subplots(1, len(sample), figsize=(5 * len(sample), 5))
    if len(sample) == 1:
        axes = [axes]

    for ax, img_id in zip(axes, sample):
        meta = id2img[img_id]
        img_path = EXTRACT_ROOT / 'buzzset' / split / meta['file_name']
        bgr = cv2.imread(str(img_path))
        if bgr is None:
            ax.set_title('Introuvable'); continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).copy()
        for ann in id2anns.get(img_id, []):
            x, y, w, h = [int(v) for v in ann['bbox']]
            c = COLORS.get(ann['category_id'], (255,255,0))
            cv2.rectangle(rgb, (x, y), (x+w, y+h), c, 2)
            cv2.putText(rgb, CLASS_NAMES.get(ann['category_id'], '?'),
                        (x, max(y-4, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, c, 1)
        ax.imshow(rgb)
        ax.set_title(meta['file_name'].split('/')[-1])
        ax.axis('off')

    plt.tight_layout()
    plt.show()

show_annotated_sample('train', n=4)

## 6 — Masques SAM 2 (optionnel)

> Génère les crops RGBA pour le copy-paste (classes rares : moth, hoverfly).  
> Nécessite `sam2` et un checkpoint. Peut être ignoré.

In [ ]:
SAM2_CKPT     = 'weights/sam2_hiera_large.pt'
MASKS_OUT     = 'data/sam2_masks/'
RUN_SAM2_STEP = False

if RUN_SAM2_STEP:
    !python scripts/extract_sam2_masks.py \
        --ann      data/buzzset/annotations/train.json \
        --img-dir  data/buzzset/train/ \
        --out-dir  {MASKS_OUT} \
        --sam2-ckpt {SAM2_CKPT} \
        --classes moth hoverfly
else:
    print('Étape SAM2 ignorée (RUN_SAM2_STEP=False)')

## 7 — Entraînement

Les configs `colab_t4` et `colab_l4` définissent automatiquement :
- batch size, gradient accumulation, amp_dtype adaptés au GPU
- `num_frames=1 / strategy=none` → supprime le chargement inutile des 5 frames contexte
- `attribute_heads.enabled=false` → économise VRAM

Des overrides ponctuels restent possibles via `clé=valeur` en fin de commande.

In [ ]:
EXPERIMENT_NAME = 'rfdetr_v1'
EPOCHS          = 50
COPY_PASTE_ON   = False

print(f'Config : {CONFIG_NAME}')
print(f'Expérience : {EXPERIMENT_NAME}  |  Epochs : {EPOCHS}')

In [ ]:
!python src/train.py \
    --config-name={CONFIG_NAME} \
    project.experiment_name={EXPERIMENT_NAME} \
    training.experiment_name={EXPERIMENT_NAME} \
    training.epochs={EPOCHS} \
    data.root=data/buzzset/ \
    data.train_ann=data/buzzset/annotations/train.json \
    data.val_ann=data/buzzset/annotations/valid.json \
    augmentations.train.copy_paste.enabled={str(COPY_PASTE_ON).lower()}

In [ ]:
import torch

ckpt_path = f'outputs/{EXPERIMENT_NAME}/best.pth'
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    print(f'Meilleur checkpoint — epoch {ckpt["epoch"]}, best_metric = {ckpt["best_metric"]:.4f}')
else:
    print('Checkpoint best.pth introuvable')

## 8 — Évaluation COCO sur la validation

In [ ]:
import torch
from omegaconf import OmegaConf
from torch.utils.data import DataLoader
from torch.amp import autocast
from tqdm.notebook import tqdm

from src.data import BuzzSetDataset
from src.data.augmentations.transforms import get_val_transforms
from src.models.rfdetr_temporal import RFDETRTemporal
from src.evaluation.metrics.coco_eval import BuzzSpotEvaluator
from src.train import collate_fn

WEIGHTS_PATH = f'outputs/{EXPERIMENT_NAME}/best.pth'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device :', DEVICE)

ckpt = torch.load(WEIGHTS_PATH, map_location=DEVICE, weights_only=False)
cfg  = OmegaConf.create(ckpt['config'])

model = RFDETRTemporal(
    num_classes=cfg.data.num_classes,
    d_model=cfg.model.hidden_dim,
    num_queries=cfg.model.num_queries,
    num_frames=cfg.data.temporal.num_frames,
    temporal_type=cfg.model.temporal_head.type,
    use_attribute_heads=cfg.model.attribute_heads.enabled,
    pretrained=False,
)
model.load_state_dict(ckpt['model_state_dict'])
model.to(DEVICE).eval()
print(f'Modèle chargé — epoch {ckpt["epoch"]}')

In [ ]:
val_ds = BuzzSetDataset(
    ann_file='data/buzzset/annotations/valid.json',
    img_dir='data/buzzset/',
    transforms=get_val_transforms(tile_size=cfg.data.tile.size),
    num_frames=cfg.data.temporal.num_frames,
    temporal_strategy=cfg.data.temporal.strategy,
    split='val',
)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, collate_fn=collate_fn)

evaluator = BuzzSpotEvaluator(
    ann_file='data/buzzset/annotations/valid.json',
    class_names=list(cfg.data.class_names),
)

use_amp = DEVICE.type == 'cuda'
with torch.no_grad():
    for batch in tqdm(val_loader, desc='Évaluation'):
        images  = batch['image'].to(DEVICE)
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in batch['target']]
        with autocast(device_type=DEVICE.type, enabled=use_amp):
            outputs = model(images)
        evaluator.update(outputs, targets)

metrics = evaluator.summarize()

print('\n' + '='*50)
print('RÉSULTATS COCO')
print('='*50)
for k, v in metrics.items():
    print(f'  {k:<30s}: {v:.4f}')

In [ ]:
cls_names = ['bee', 'bumblebee', 'hoverfly', 'moth']
ap5095 = [metrics.get(f'AP_{c}_50_95', 0) for c in cls_names]
ap50   = [metrics.get(f'AP_{c}_50',    0) for c in cls_names]

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(cls_names))
w = 0.35
ax.bar(x - w/2, ap5095, w, label='AP@0.5:0.95', color='steelblue')
ax.bar(x + w/2, ap50,   w, label='AP@0.5',      color='lightblue')
ax.set_xticks(x)
ax.set_xticklabels(cls_names)
ax.set_ylabel('Average Precision')
ax.set_title(f'Per-class AP — mAP@0.5:0.95 = {metrics["mAP_50_95"]:.4f}')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 9 — Inférence sur le jeu de test

In [ ]:
PREDICTIONS = f'submissions/predictions_{EXPERIMENT_NAME}.json'
SCORE_THR   = 0.3

!python scripts/infer.py \
    --weights   {WEIGHTS_PATH} \
    --test-ann  data/buzzset/annotations/test.json \
    --test-dir  data/buzzset/test/ \
    --out       {PREDICTIONS} \
    --score-thr {SCORE_THR} \
    --zip

In [ ]:
from pathlib import Path

pred_path = Path(PREDICTIONS)
if pred_path.exists():
    with open(pred_path) as f:
        preds = json.load(f)
    print(f'Total prédictions : {len(preds)}')
    cls_counts = {}
    for p in preds:
        cls_counts[p['category_id']] = cls_counts.get(p['category_id'], 0) + 1
    for cat_id, cnt in sorted(cls_counts.items()):
        print(f'  {CLASS_NAMES.get(cat_id, cat_id):12s} : {cnt:6d}')
    scores = [p['score'] for p in preds]
    print(f'Score — min: {min(scores):.3f}, mean: {np.mean(scores):.3f}, max: {max(scores):.3f}')
else:
    print('Fichier de prédictions introuvable')

## 10 — Génération de la soumission Codabench

In [ ]:
import zipfile
from pathlib import Path

pred_json = Path(PREDICTIONS)
zip_path  = pred_json.parent / (pred_json.stem + '.zip')

if pred_json.exists():
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_json, 'predictions.json')
    print(f'Soumission prête : {zip_path}  ({zip_path.stat().st_size / 1024:.1f} KB)')
    print('Uploader ce fichier sur Codabench.')

    # Téléchargement automatique dans Colab
    if IN_COLAB:
        from google.colab import files
        files.download(str(zip_path))
else:
    print('Fichier de prédictions manquant')

## 11 — Pseudo-labeling + self-training (optionnel)

> Nécessite Grounding DINO + SAM 2. Peut être ignoré.

In [ ]:
RUN_PSEUDO_LABELING = False

if RUN_PSEUDO_LABELING:
    GDINO_CFG  = 'weights/GroundingDINO_SwinT_OGC.py'
    GDINO_CKPT = 'weights/groundingdino_swint_ogc.pth'
    SAM2_CKPT  = 'weights/sam2_hiera_large.pt'

    !python scripts/pseudo_label.py \
        --weights    {WEIGHTS_PATH} \
        --test-ann   data/buzzset/annotations/test.json \
        --test-dir   data/buzzset/test/ \
        --gdino-cfg  {GDINO_CFG} \
        --gdino-ckpt {GDINO_CKPT} \
        --sam2-ckpt  {SAM2_CKPT} \
        --out-ann    data/buzzset/annotations/pseudo_train.json \
        --conf       0.5 \
        --epochs     2
else:
    print('Pseudo-labeling ignoré (RUN_PSEUDO_LABELING=False)')